In [25]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MMDS") \
    .master("local[*]") \
    .getOrCreate()

## Users

In [ ]:
from pyspark.sql.functions import col

users_df = spark.read.option("delimiter", "::").csv("../data/users.dat")
users_df = users_df.toDF(
    "user_id",
    "gender",
    "age",
    "occupation",
    "zip_code",
)

users = (
    users_df
    .withColumn("user_id", col("user_id").cast("int"))
    .withColumn("age", col("age").cast("int"))
    .withColumn("occupation", col("occupation").cast("int"))
)

users.printSchema()
users.show()

root
 |-- user_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- occupation: integer (nullable = true)
 |-- zip_code: string (nullable = true)

+-------+------+---+----------+--------+
|user_id|gender|age|occupation|zip_code|
+-------+------+---+----------+--------+
|      1|     F|  1|        10|   48067|
|      2|     M| 56|        16|   70072|
|      3|     M| 25|        15|   55117|
|      4|     M| 45|         7|   02460|
|      5|     M| 25|        20|   55455|
|      6|     F| 50|         9|   55117|
|      7|     M| 35|         1|   06810|
|      8|     M| 25|        12|   11413|
|      9|     M| 25|        17|   61614|
|     10|     F| 35|         1|   95370|
|     11|     F| 25|         1|   04093|
|     12|     M| 25|        12|   32793|
|     13|     M| 45|         1|   93304|
|     14|     M| 35|         0|   60126|
|     15|     M| 25|         7|   22903|
|     16|     F| 35|         0|   20670|
|     17|     M| 5

In [27]:
print(f"There are {users.count()} users")

There are 6040 users


In [28]:
from pyspark.sql.functions import sum, when

users.select([
    sum(when(col(c).isNull(), 1)).alias(c) for c in users.columns
]).show()

+-------+------+----+----------+--------+
|user_id|gender| age|occupation|zip_code|
+-------+------+----+----------+--------+
|   NULL|  NULL|NULL|      NULL|    NULL|
+-------+------+----+----------+--------+



In [29]:
users.groupBy("gender").count().orderBy("count", ascending=False).show()

+------+-----+
|gender|count|
+------+-----+
|     M| 4331|
|     F| 1709|
+------+-----+



In [30]:
users.groupBy("age").count().orderBy("count", ascending=False).show()

+---+-----+
|age|count|
+---+-----+
| 25| 2096|
| 35| 1193|
| 18| 1103|
| 45|  550|
| 50|  496|
| 56|  380|
|  1|  222|
+---+-----+



Age '1' is probably for users who decided not to answer. Maybe should be handeled somehow so that it's not treated as an entry in an ordinal field.

### Occupation

In [31]:
occupation_data = [
    (0,  "other"),
    (1,  "academic/educator"),
    (2,  "artist"),
    (3,  "clerical/admin"),
    (4,  "college/grad student"),
    (5,  "customer service"),
    (6,  "doctor/health care"),
    (7,  "executive/managerial"),
    (8,  "farmer"),
    (9,  "homemaker"),
    (10, "K-12 student"),
    (11, "lawyer"),
    (12, "programmer"),
    (13, "retired"),
    (14, "sales/marketing"),
    (15, "scientist"),
    (16, "self-employed"),
    (17, "technician/engineer"),
    (18, "tradesman/craftsman"),
    (19, "unemployed"),
    (20, "writer"),
]

occupation_df = spark.createDataFrame(
    occupation_data,
    ["occupation", "occupation_name"]
)

occupation_df.show()

+----------+--------------------+
|occupation|     occupation_name|
+----------+--------------------+
|         0|               other|
|         1|   academic/educator|
|         2|              artist|
|         3|      clerical/admin|
|         4|college/grad student|
|         5|    customer service|
|         6|  doctor/health care|
|         7|executive/managerial|
|         8|              farmer|
|         9|           homemaker|
|        10|        K-12 student|
|        11|              lawyer|
|        12|          programmer|
|        13|             retired|
|        14|     sales/marketing|
|        15|           scientist|
|        16|       self-employed|
|        17| technician/engineer|
|        18| tradesman/craftsman|
|        19|          unemployed|
+----------+--------------------+
only showing top 20 rows


In [32]:
users_enriched = users.join(occupation_df, on='occupation', how='left')
users_enriched.show()

+----------+-------+------+---+--------+--------------------+
|occupation|user_id|gender|age|zip_code|     occupation_name|
+----------+-------+------+---+--------+--------------------+
|         0|     14|     M| 35|   60126|               other|
|         0|     16|     F| 35|   20670|               other|
|         7|      4|     M| 45|   02460|executive/managerial|
|         7|     15|     M| 25|   22903|executive/managerial|
|         9|      6|     F| 50|   55117|           homemaker|
|        17|      9|     M| 25|   61614| technician/engineer|
|         1|      7|     M| 35|   06810|   academic/educator|
|         1|     10|     F| 35|   95370|   academic/educator|
|         1|     11|     F| 25|   04093|   academic/educator|
|         1|     13|     M| 45|   93304|   academic/educator|
|         1|     17|     M| 50|   95350|   academic/educator|
|        10|      1|     F|  1|   48067|        K-12 student|
|        10|     19|     M|  1|   48073|        K-12 student|
|       

In [33]:
users_enriched.groupBy("occupation_name").count().orderBy("count", ascending=False).show()

+--------------------+-----+
|     occupation_name|count|
+--------------------+-----+
|college/grad student|  759|
|               other|  711|
|executive/managerial|  679|
|   academic/educator|  528|
| technician/engineer|  502|
|          programmer|  388|
|     sales/marketing|  302|
|              writer|  281|
|              artist|  267|
|       self-employed|  241|
|  doctor/health care|  236|
|        K-12 student|  195|
|      clerical/admin|  173|
|           scientist|  144|
|             retired|  142|
|              lawyer|  129|
|    customer service|  112|
|           homemaker|   92|
|          unemployed|   72|
| tradesman/craftsman|   70|
+--------------------+-----+
only showing top 20 rows


In [34]:
users.groupBy("gender", "age").count().orderBy("count", ascending=False).show()

+------+---+-----+
|gender|age|count|
+------+---+-----+
|     M| 25| 1538|
|     M| 35|  855|
|     M| 18|  805|
|     F| 25|  558|
|     M| 45|  361|
|     M| 50|  350|
|     F| 35|  338|
|     F| 18|  298|
|     M| 56|  278|
|     F| 45|  189|
|     F| 50|  146|
|     M|  1|  144|
|     F| 56|  102|
|     F|  1|   78|
+------+---+-----+



In [35]:
users_enriched.groupBy("gender", "occupation_name").count().orderBy("count", ascending=False).show()

+------+--------------------+-----+
|gender|     occupation_name|count|
+------+--------------------+-----+
|     M|executive/managerial|  540|
|     M|college/grad student|  525|
|     M|               other|  479|
|     M| technician/engineer|  450|
|     M|          programmer|  338|
|     M|   academic/educator|  319|
|     F|college/grad student|  234|
|     F|               other|  232|
|     M|     sales/marketing|  223|
|     F|   academic/educator|  209|
|     M|              writer|  203|
|     M|       self-employed|  190|
|     M|              artist|  176|
|     F|executive/managerial|  139|
|     M|  doctor/health care|  134|
|     M|        K-12 student|  129|
|     M|           scientist|  116|
|     M|             retired|  108|
|     M|              lawyer|  107|
|     F|  doctor/health care|  102|
+------+--------------------+-----+
only showing top 20 rows


In [36]:
users.groupBy("user_id").count().filter("count > 1").show()

+-------+-----+
|user_id|count|
+-------+-----+
+-------+-----+



## Movies

In [ ]:
from pyspark.sql.functions import split, substring

movies_df = spark.read.option("delimiter", "::").csv("../data/movies.dat")
movies_df = movies_df.toDF(
    "movie_id",
    "title",
    "genres",
)

movies = (
    movies_df
    .withColumn("movie_id", col("movie_id").cast("int"))
    .withColumn("genres", split("genres", r"\|"))
    .withColumn("year", substring("title", -5, 4).cast("int"))
)

movies.show(truncate=False)
movies.printSchema()

+--------+-------------------------------------+--------------------------------+----+
|movie_id|title                                |genres                          |year|
+--------+-------------------------------------+--------------------------------+----+
|1       |Toy Story (1995)                     |[Animation, Children's, Comedy] |1995|
|2       |Jumanji (1995)                       |[Adventure, Children's, Fantasy]|1995|
|3       |Grumpier Old Men (1995)              |[Comedy, Romance]               |1995|
|4       |Waiting to Exhale (1995)             |[Comedy, Drama]                 |1995|
|5       |Father of the Bride Part II (1995)   |[Comedy]                        |1995|
|6       |Heat (1995)                          |[Action, Crime, Thriller]       |1995|
|7       |Sabrina (1995)                       |[Comedy, Romance]               |1995|
|8       |Tom and Huck (1995)                  |[Adventure, Children's]         |1995|
|9       |Sudden Death (1995)              

In [38]:
print(f"There are {movies.count()} movies")

There are 3883 movies


In [39]:
from pyspark.sql.functions import sum, when

movies.select([
    sum(when(col(c).isNull(), 1)).alias(c) for c in movies.columns
]).show()

+--------+-----+------+----+
|movie_id|title|genres|year|
+--------+-----+------+----+
|    NULL| NULL|  NULL|NULL|
+--------+-----+------+----+



In [40]:
movies.groupBy("movie_id").count().filter("count > 1").show()

+--------+-----+
|movie_id|count|
+--------+-----+
+--------+-----+



In [41]:
movies.groupBy("year").count().summary().show()

+-------+------------------+-----------------+
|summary|              year|            count|
+-------+------------------+-----------------+
|  count|                81|               81|
|   mean|1959.9382716049383|47.93827160493827|
| stddev|23.628555647252515|81.78635975500627|
|    min|              1919|                1|
|    25%|              1940|               11|
|    50%|              1960|               19|
|    75%|              1980|               35|
|    max|              2000|              345|
+-------+------------------+-----------------+



In [42]:
movies.filter("year is NULL").count()

0

## Ratings

In [ ]:
from pyspark.sql.functions import col, from_unixtime

ratings_df = spark.read.option("delimiter", "::").csv("../data/ratings.dat")
ratings_df = ratings_df.toDF(
    "user_id",
    "movie_id",
    "rating",
    "timestamp",
)

ratings = (
    ratings_df
    .withColumn("user_id", col("user_id").cast("int"))
    .withColumn("movie_id", col("movie_id").cast("int"))
    .withColumn("rating", col("rating").cast("int"))
    .withColumn("timestamp", from_unixtime(col("timestamp")).cast("timestamp"))
)

ratings.printSchema()
ratings.show()

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)

+-------+--------+------+-------------------+
|user_id|movie_id|rating|          timestamp|
+-------+--------+------+-------------------+
|      1|    1193|     5|2001-01-01 00:12:40|
|      1|     661|     3|2001-01-01 00:35:09|
|      1|     914|     3|2001-01-01 00:32:48|
|      1|    3408|     4|2001-01-01 00:04:35|
|      1|    2355|     5|2001-01-07 01:38:11|
|      1|    1197|     3|2001-01-01 00:37:48|
|      1|    1287|     5|2001-01-01 00:33:59|
|      1|    2804|     5|2001-01-01 00:11:59|
|      1|     594|     4|2001-01-01 00:37:48|
|      1|     919|     4|2001-01-01 00:22:48|
|      1|     595|     5|2001-01-07 01:37:48|
|      1|     938|     4|2001-01-01 00:29:12|
|      1|    2398|     4|2001-01-01 00:38:01|
|      1|    2918|     4|2001-01-01 00:35:24|
|      1|    1035|     5|2001-01-01 00:29:13|
| 

In [58]:
ratings.summary().show()

+-------+------------------+------------------+------------------+
|summary|           user_id|          movie_id|            rating|
+-------+------------------+------------------+------------------+
|  count|           1000209|           1000209|           1000209|
|   mean| 3024.512347919285|1865.5398981612843| 3.581564453029317|
| stddev|1728.4126948999715|1096.0406894572482|1.1171018453732606|
|    min|                 1|                 1|                 1|
|    25%|              1505|              1030|                 3|
|    50%|              3071|              1835|                 4|
|    75%|              4476|              2770|                 4|
|    max|              6040|              3952|                 5|
+-------+------------------+------------------+------------------+



In [ ]:
ratings.groupBy("user_id").count().orderBy("count", ascending=False).show()

+-------+-----+
|user_id|count|
+-------+-----+
|   4169| 2314|
|   1680| 1850|
|   4277| 1743|
|   1941| 1595|
|   1181| 1521|
|    889| 1518|
|   3618| 1344|
|   2063| 1323|
|   1150| 1302|
|   1015| 1286|
|   5795| 1277|
|   4344| 1271|
|   1980| 1260|
|   2909| 1258|
|   1449| 1243|
|   4510| 1240|
|    424| 1226|
|   4227| 1222|
|   5831| 1220|
|   3841| 1216|
+-------+-----+
only showing top 20 rows


In [51]:
ratings.groupBy("user_id").count().summary().show()

+-------+-----------------+------------------+
|summary|          user_id|             count|
+-------+-----------------+------------------+
|  count|             6040|              6040|
|   mean|           3020.5| 165.5975165562914|
| stddev|1743.742144546223|192.74702906977765|
|    min|                1|                20|
|    25%|             1509|                44|
|    50%|             3019|                95|
|    75%|             4529|               207|
|    max|             6040|              2314|
+-------+-----------------+------------------+



In [52]:
ratings.groupBy("movie_id").count().orderBy("count", ascending=False).show()

+--------+-----+
|movie_id|count|
+--------+-----+
|    2858| 3428|
|     260| 2991|
|    1196| 2990|
|    1210| 2883|
|     480| 2672|
|    2028| 2653|
|     589| 2649|
|    2571| 2590|
|    1270| 2583|
|     593| 2578|
|    1580| 2538|
|    1198| 2514|
|     608| 2513|
|    2762| 2459|
|     110| 2443|
|    2396| 2369|
|    1197| 2318|
|     527| 2304|
|    1617| 2288|
|    1265| 2278|
+--------+-----+
only showing top 20 rows


In [53]:
ratings.groupBy("movie_id").count().summary().show()

+-------+------------------+------------------+
|summary|          movie_id|             count|
+-------+------------------+------------------+
|  count|              3706|              3706|
|   mean|1995.5731246627092|269.88909875876953|
| stddev|1151.1480449998355| 384.0478375720254|
|    min|                 1|                 1|
|    25%|               989|                33|
|    50%|              2033|               123|
|    75%|              2991|               350|
|    max|              3952|              3428|
+-------+------------------+------------------+



In [71]:
from pyspark.sql.functions import year, month, hour

ratings.withColumn("year", year(col("timestamp"))).groupBy("year").count().orderBy("count", ascending=False).show()

+----+------+
|year| count|
+----+------+
|2000|904551|
|2001| 68263|
|2002| 24043|
|2003|  3352|
+----+------+



In [70]:
ratings.withColumn("month", month(col("timestamp"))).groupBy("month").count().orderBy("count", ascending=False).show()


+-----+------+
|month| count|
+-----+------+
|   11|294878|
|    8|189304|
|   12|118680|
|    7| 96909|
|    5| 73435|
|    6| 61277|
|    9| 56318|
|   10| 46210|
|    1| 23271|
|    4| 19371|
|    2| 12138|
|    3|  8418|
+-----+------+



In [73]:
ratings.withColumn("hour", hour(col("timestamp"))).groupBy("hour").count().orderBy("count", ascending=False).show(n=24)

+----+-----+
|hour|count|
+----+-----+
|  23|65440|
|   5|62522|
|  22|61657|
|   3|60542|
|   0|59842|
|   6|59564|
|  21|59395|
|   4|56942|
|  20|54422|
|  19|51123|
|   1|50780|
|   2|50571|
|   7|49562|
|  18|45601|
|   8|41143|
|  17|33364|
|   9|31011|
|  16|25899|
|  10|24381|
|  15|16266|
|  11|15565|
|  12| 9298|
|  14| 8209|
|  13| 7110|
+----+-----+



In [75]:
ratings.withColumn("year", year("timestamp")).groupBy("year").avg('rating').orderBy("year").show()

+----+-----------------+
|year|      avg(rating)|
+----+-----------------+
|2000|3.590338189886474|
|2001|  3.5132062757277|
|2002|3.459094123029572|
|2003|3.484486873508353|
+----+-----------------+



In [93]:
from pyspark.sql.functions import explode

ratings_with_genres = (
    ratings
    .join(movies, on="movie_id", how="inner")
    .withColumn("genre", explode("genres"))
)

ratings_with_genres.show()

+--------+-------+------+-------------------+--------------------+--------------------+----+----------+
|movie_id|user_id|rating|          timestamp|               title|              genres|year|     genre|
+--------+-------+------+-------------------+--------------------+--------------------+----+----------+
|    1193|      1|     5|2001-01-01 00:12:40|One Flew Over the...|             [Drama]|1975|     Drama|
|     661|      1|     3|2001-01-01 00:35:09|James and the Gia...|[Animation, Child...|1996| Animation|
|     661|      1|     3|2001-01-01 00:35:09|James and the Gia...|[Animation, Child...|1996|Children's|
|     661|      1|     3|2001-01-01 00:35:09|James and the Gia...|[Animation, Child...|1996|   Musical|
|     914|      1|     3|2001-01-01 00:32:48| My Fair Lady (1964)|  [Musical, Romance]|1964|   Musical|
|     914|      1|     3|2001-01-01 00:32:48| My Fair Lady (1964)|  [Musical, Romance]|1964|   Romance|
|    3408|      1|     4|2001-01-01 00:04:35|Erin Brockovich (..

In [109]:
from pyspark.sql.functions import count, avg
top10_most_rated_movies = (
    ratings_with_genres
        .groupBy('movie_id', "title")
        .agg(
            count("*").alias("num_ratings"),
            avg("rating").alias("avg_rating")
        )
        .orderBy("num_ratings", ascending=False)
        .limit(10)
)

top10_most_rated_movies.show(truncate=False)

+--------+-----------------------------------------------------+-----------+------------------+
|movie_id|title                                                |num_ratings|avg_rating        |
+--------+-----------------------------------------------------+-----------+------------------+
|1196    |Star Wars: Episode V - The Empire Strikes Back (1980)|14950      |4.292976588628763 |
|1210    |Star Wars: Episode VI - Return of the Jedi (1983)    |14415      |4.022892819979188 |
|260     |Star Wars: Episode IV - A New Hope (1977)            |11964      |4.453694416583082 |
|1580    |Men in Black (1997)                                  |10152      |3.739952718676123 |
|1197    |Princess Bride, The (1987)                           |9272       |4.3037100949094045|
|1617    |L.A. Confidential (1997)                             |9152       |4.219405594405594 |
|1097    |E.T. the Extra-Terrestrial (1982)                    |9076       |3.9651828999559275|
|2628    |Star Wars: Episode I - The Pha

In [110]:
from pyspark.sql.functions import expr

top_movies_weighted = (
    ratings
    .join(movies, "movie_id")
    .groupBy("movie_id", "title")
    .agg(
        count("*").alias("num_ratings"),
        avg("rating").alias("avg_rating")
    )
    .withColumn(
        "score",
        expr("avg_rating * log10(num_ratings)")
    )
    .orderBy("score", ascending=False)
    .limit(10)
)

top_movies_weighted.show(truncate=False)

+--------+-----------------------------------------------------+-----------+------------------+------------------+
|movie_id|title                                                |num_ratings|avg_rating        |score             |
+--------+-----------------------------------------------------+-----------+------------------+------------------+
|260     |Star Wars: Episode IV - A New Hope (1977)            |2991       |4.453694416583082 |15.480224151785418|
|2858    |American Beauty (1999)                               |3428       |4.3173862310385065|15.262136533289725|
|318     |Shawshank Redemption, The (1994)                     |2227       |4.554557700942973 |15.247384895094859|
|1198    |Raiders of the Lost Ark (1981)                       |2514       |4.477724741447892 |15.225899714439246|
|527     |Schindler's List (1993)                              |2304       |4.510416666666667 |15.166196995492276|
|858     |Godfather, The (1972)                                |2223       |4.52

In [94]:
from pyspark.sql.functions import avg, count

top_genres_by_rating = (
    ratings_with_genres
    .groupBy("genre")
    .agg(
        avg("rating").alias("avg_rating"),
        count("*").alias("num_ratings")
    )
    .orderBy("avg_rating", ascending=False)
)

top_genres_by_rating.show(truncate=False)

+-----------+------------------+-----------+
|genre      |avg_rating        |num_ratings|
+-----------+------------------+-----------+
|Film-Noir  |4.075187558184108 |18261      |
|Documentary|3.933122629582807 |7910       |
|War        |3.893326717935996 |68527      |
|Drama      |3.766332232342065 |354529     |
|Crime      |3.708678543141273 |79541      |
|Animation  |3.684868223500335 |43293      |
|Mystery    |3.6681019463387923|40178      |
|Musical    |3.6655189849035708|41533      |
|Western    |3.6377701493980563|20683      |
|Romance    |3.607464598740535 |147523     |
|Thriller   |3.5704660480809784|189680     |
|Comedy     |3.522098827752538 |356580     |
|Action     |3.4911849357368414|257457     |
|Adventure  |3.477256948332624 |133953     |
|Sci-Fi     |3.466521291339784 |157294     |
|Fantasy    |3.447370595851354 |36301      |
|Children's |3.422034743579087 |72186      |
|Horror     |3.215013222318226 |76386      |
+-----------+------------------+-----------+



In [111]:
top10_most_rated_genres = (
    ratings_with_genres
    .groupBy("genre")
    .agg(
        avg("rating").alias("avg_rating"),
        count("*").alias("num_ratings")
    )
    .orderBy("num_ratings", ascending=False)
    .limit(10)
)

top10_most_rated_genres.show(truncate=False)

+----------+------------------+-----------+
|genre     |avg_rating        |num_ratings|
+----------+------------------+-----------+
|Comedy    |3.522098827752538 |356580     |
|Drama     |3.766332232342065 |354529     |
|Action    |3.4911849357368414|257457     |
|Thriller  |3.5704660480809784|189680     |
|Sci-Fi    |3.466521291339784 |157294     |
|Romance   |3.607464598740535 |147523     |
|Adventure |3.477256948332624 |133953     |
|Crime     |3.708678543141273 |79541      |
|Horror    |3.215013222318226 |76386      |
|Children's|3.422034743579087 |72186      |
+----------+------------------+-----------+

